# IS477 Final Project

In [1]:
import pandas as pd
import requests
from pathlib import Path
import hashlib
import gzip
import io
import json

In [2]:
## loading huggingface csv

huggingface = pd.read_csv('huggingface_spotify.csv')
hf_sorted = huggingface.sort_values(by=['artists']).reset_index()
hf_artists = pd.DataFrame({'artists':huggingface['artists'].unique()}).sort_values(by=['artists']).reset_index()['artists']
hf_artists

0                                                   !nvite
1                                    "Puppy Dog Pals" Cast
2                                      "Weird Al" Yankovic
3        #Kids;Nursery Rhymes;Nursery Rhymes and Kids S...
4                                                   $affie
                               ...                        
31433                                                   黃妃
31434                                                  黃小琥
31435                                                  黃敏華
31436                                              龍藏Ryuzo
31437                                                  NaN
Name: artists, Length: 31438, dtype: object

In [39]:
## Querying the MusicBrainz API ## (can try pagination later)

url = 'http://musicbrainz.org/ws/2/release'
for artist in hf_artists[15:]:
    params = {"fmt":"json", "limit":1000, "query":f"{artist}"}

    res = requests.get(url, params=params, headers={'User-Agent':'ISProject/1.0 (sciencescholar14@gmail.com)'})
    res.raise_for_status()
    data = res.json()

    with open('musicbrainz.json', 'a') as f:
        f.write(json.dumps(data, indent=3, sort_keys=True))
    print(artist)

HTTPError: 503 Server Error: Service Temporarily Unavailable for url: http://musicbrainz.org/ws/2/release?fmt=json&limit=1000&query=04+Limited+Sazabys

In [3]:
hf_query_params = huggingface[['track_id', 'artists', 'album_name', 'track_name']]

def set_p_delim(df):
    return str(df['album_name']) + '|'

def set_p_list(df):
    return list(set(str(df['album_name']).split('|')[:-1]))

hf_query_params['album_name'] = hf_query_params.apply(set_p_delim, axis=1)
hf_query_params = hf_query_params.groupby('artists').agg('sum').reset_index()
hf_query_params['album_name'] = hf_query_params.apply(set_p_list, axis=1)
hf_query_params

C:\Users\Jonah\AppData\Local\Temp\ipykernel_19128\2142246194.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hf_query_params['album_name'] = hf_query_params.apply(set_p_delim, axis=1)


,artists,track_id,album_name,track_name
0,!nvite,1d4ZvL8uuUPTEAnocC3zEa3v6ypsJzaoY2xgYp6mMJfM,"[pagadoff, strolling]",strollingpagadoff
1,"""Puppy Dog Pals"" Cast",5qtlopq4SnnvVeiQVt3M0n4aY2hh55axhL2qYYqXNoOM,[Puppy Dog Pals: Disney Junior Music],Puppy Dog Pals Main Title ThemeGoing on a Mission
2,"""Weird Al"" Yankovic",64vs4dpa55X1Z2thSiK3Ad5r96TaQquRrlo3Ym3ZlSL23i...,"[""Weird Al"" Yankovic, Bad Hair Day, Running Wi...","Grapefruit Diet (Parody of ""Zoot Suit Riot"" by..."
3,#Kids;Nursery Rhymes;Nursery Rhymes and Kids S...,5v8LWqqEVnbwCFGqp7jVXP,[Nursery Rhymes For Young Kids],Wheels on The Bus Go Round and Round (Instrume...
4,$affie,4ufMIn7ee8xlsAePOf22r8,[summertime],summertime
...,...,...,...,...
31432,黃仲崑,0k63CZovrFDeECSiNxytQx33A2M24YU78cU50ODE1srp5N...,"[經典復刻21黃仲崑, 記得我]",牽掛你說過記得我
31433,黃妃,0M4N45TMVz7Py2d6tEfDRX0xCXMZlt1QfWfhtuTIFBpk,"[水水水, 非常妃]",恨世生溫暖的所在
31434,黃小琥,4xZIMRwaaBx7CZMmM6KLuh,[簡單不簡單],沒那麽簡單
31435,黃敏華,1Q5d3X55pI7nBXYAdZ0g8Z,[關不掉的聲音],堤岸


In [65]:
## querying last.fm API ## (complete structure for single query (sample query below) -- not sure if can bulk query from last.fm)

with open('lastfmapi_key.txt') as f:
    apikey = f.readline()

fm_url = ' http://ws.audioscrobbler.com/2.0/'
for i in range(len(hf_artists)):
    for album in hf_query_params['album_name'][i]:
# for page in range(1, 11):
        params2 = {'api_key':apikey, 'format':'json', 'method':'album.getInfo', 'artist':f'{hf_artists[i]}', 'album':f'{album}', 'limit':50}
        response = requests.get(fm_url, params=params2, headers={'User-Agent':'ISProject/1.0 (sciencescholar14@gmail.com)'})
        data = response.json()
        with open('lastfm.json', 'a') as f: ## writing paginated query results ##
            f.write(json.dumps(data, indent=3, sort_keys=True))

KeyboardInterrupt: 